# Dev29 - Single Wavelength Greyscale Analysis

This notebook visualizes single wavelength intensities as greyscale heatmaps with derivative analysis.

**Workflow:**
1. Load all 5 files from transect 028
2. Apply preprocessing (illumination correction, wavelength filtering, smoothing, normalization)
3. Plot single wavelength intensities as greyscale heatmaps
4. Apply derivative analysis (0th, 1st, 2nd order)

**Target Wavelengths:**
- 500 nm: Reference green, comparison baseline
- 550 nm: Green peak, biofilm/algae vs metal
- 580 nm: Sediment/iron stains starting point
- 600 nm: Main rust band, oxidized metal
- 620 nm: Cyanobacteria, reddish leakage around UXO
- 675 nm: Chlorophyll absorption (biofilm detection)

**Derivative Analysis:**
- 0th derivative: Raw intensity
- 1st derivative: Rate of change (slope)
- 2nd derivative: Curvature (absorption features)

**Date:** November 3, 2025

## Setup and Import

In [ ]:
# Import required libraries
import importlib
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Add parent directory to path
sys.path.append(os.path.abspath("../"))

# Import gref4hsi modules
from utils.gref_pipeline import georef
from gref_pipeline import config

importlib.reload(georef)
from utils.gref_pipeline.georef import *

# Import NDI analysis utilities (now includes wavelength visualization functions)
import utils.ndi_analysis_utils as ndi_utils

importlib.reload(ndi_utils)
from utils.ndi_analysis_utils import *

print("✅ All modules loaded successfully!")
print("✅ Wavelength visualization functions loaded from ndi_analysis_utils")

## Test Wavelength-to-RGB Conversion

Quick test to verify the wavelength color conversion works correctly.

In [ ]:
# Test wavelength-to-RGB conversion
test_wavelengths = [500, 550, 580, 600, 620, 660, 675]
print("Testing wavelength-to-RGB conversion:")
print("-" * 50)
for wl in test_wavelengths:
    rgb = wavelength_to_rgb(wl)
    print(f"{wl} nm → RGB: ({rgb[0]:.3f}, {rgb[1]:.3f}, {rgb[2]:.3f})")

# Test colormap creation
print("\n✅ Colormap creation test:")
cmap_test = create_wavelength_colormap(660)
print(f"   Created colormap for 660 nm (rust red)")
print(f"   Colormap name: {cmap_test.name}")

## 1. Load Transect Data

Load all 5 files from transect 028.

In [ ]:
# Load 028 transect
transect = load_transect(r"E:\mjosa_new_oct_2025\use_gref4hsi\028\output")
transect.list_files()

# Select all 5 files from transect 028
cube = transect.select_files(
    [
        "rad_uhi_20241029_125028_1",
        "rad_uhi_20241029_125028_2",
        "rad_uhi_20241029_125028_3",
        "rad_uhi_20241029_125028_4",
        "rad_uhi_20241029_125028_5",
    ]
)
# cube.describe()

## 2. Preprocessing Pipeline

Apply preprocessing steps to normalize the spectral data.

### 2.1 Apply Illumination Correction

In [ ]:
# Apply illumination correction
cube.apply_illumination_correction_v2()

### 2.2 Visualize Transect (RGB)

In [ ]:
# Plot RGB to visualize the full transect
cube.plot_rgb(use_corrected=True, figsize=(20, 6))

### 2.2b Test RGB with Derivatives

New feature: Visualize RGB composite using derivatives instead of raw intensity.

In [ ]:
# # Test: RGB with 1st derivative (rate of spectral change)
# cube.plot_rgb(
#     use_corrected=True, figsize=(20, 6), derivative_order=1, derivative_window=2
# )

In [ ]:
# # Test: RGB with 2nd derivative (curvature of spectral features)
# cube.plot_rgb(
#     use_corrected=True, figsize=(20, 6), derivative_order=2, derivative_window=2
# )

In [ ]:
cube.apply_spectral_smoothing(method="gaussian", gaussian_sigma=5.0)

### 2.3 Crop Wavelengths (490-680 nm)

In [ ]:
# Crop wavelengths to 490-680 nm
cube.apply_wavelength_filter(wavelength_range=(490, 700))

print(f"✅ Wavelength cropping complete")
print(f"   Cropped shape: {cube.data_corrected.shape}")
print(f"   Wavelength range: {cube.wavelengths[0]:.1f} - {cube.wavelengths[-1]:.1f} nm")

### 2.4 Spectral Smoothing (Moving Average, Window=10)

In [ ]:
# # Apply spectral smoothing with moving average (window=10)
# # cube.apply_spectral_smoothing(wavelength_smoothing=10, method="moving_average")
# cube.apply_spectral_smoothing(method="gaussian", gaussian_sigma=3.0)

# print(f"✅ Spectral smoothing complete")
# print(f"   Shape after smoothing: {cube.data_corrected.shape}")

### 2.5 L2 Normalization

In [ ]:
# Apply L2 normalization (unit-length spectra)
cube.apply_spectral_normalization(method="l2")

print(f"✅ L2 normalization complete")
print(
    f"   Value range: [{cube.data_corrected.min():.4f}, {cube.data_corrected.max():.4f}]"
)

In [ ]:
stop

## 3. Wavelength Visualization Functions

All wavelength visualization functions have been moved to `utils/ndi_analysis_utils.py`:
- `wavelength_to_rgb()`: Convert wavelength to RGB color
- `create_wavelength_colormap()`: Create white-to-color colormap
- `plot_single_wavelength_with_color()`: Plot single wavelength with natural color
- `plot_wavelength_grid_with_colors()`: Grid comparison with wavelength colors

## 4. Visualize Single Wavelengths with Natural Colors

Plot wavelength-specific heatmaps where:
- **White** = minimum intensity
- **Wavelength color** = maximum intensity

This creates intuitive visualizations where color intensity matches spectral intensity.

### 4.1 Reference Green - 500 nm (0th derivative)

In [ ]:
# 500 nm - Reference green baseline
intensity_500 = plot_single_wavelength_with_color(
    cube.data_corrected,
    cube.wavelengths,
    target_wavelength=500,
    derivative_order=0,
    title="500 nm - Reference Green (Raw Intensity)\nBaseline for comparison",
)

### 4.2 Green Peak - 550 nm (0th derivative)

In [ ]:
# 550 nm - Green peak (biofilm/algae vs metal)
intensity_550 = plot_single_wavelength_with_color(
    cube.data_corrected,
    cube.wavelengths,
    target_wavelength=550,
    derivative_order=0,
    title="550 nm - Green Peak (Raw Intensity)\nBiofilm/Algae vs Metal",
)

### 4.3 Sediment/Iron Stains - 580 nm (0th derivative)

In [ ]:
# 580 nm - Sediment/iron stains starting point
intensity_580 = plot_single_wavelength_with_color(
    cube.data_corrected,
    cube.wavelengths,
    target_wavelength=580,
    derivative_order=0,
    title="580 nm - Sediment/Iron Stains (Raw Intensity)\nTransition wavelength",
)

### 4.4 Rust Band - 600 nm (0th derivative)

In [ ]:
# 600 nm - Main rust band (0th derivative)
intensity_600_raw = plot_single_wavelength_with_color(
    cube.data_corrected,
    cube.wavelengths,
    target_wavelength=600,
    derivative_order=0,
    title="600 nm - Rust Band (Raw Intensity)\nOxidized metal detection",
)

### 4.5 Rust Band - 600 nm (2nd derivative)

In [ ]:
# 600 nm - Main rust band (2nd derivative for curvature)
intensity_600_2nd = plot_single_wavelength_with_color(
    cube.data_corrected,
    cube.wavelengths,
    target_wavelength=600,
    derivative_order=2,
    window=2,
    title="600 nm - Rust Band (2nd Derivative)\nCurvature analysis for oxidized metal",
)

### 4.6 Cyanobacteria/UXO Leakage - 620 nm (0th derivative)

In [ ]:
# 620 nm - Cyanobacteria and reddish leakage around UXO
intensity_620 = plot_single_wavelength_with_color(
    cube.data_corrected,
    cube.wavelengths,
    target_wavelength=620,
    derivative_order=0,
    title="620 nm - Cyanobacteria/UXO Leakage (Raw Intensity)\nReddish signatures",
)

### 4.7 **IMPORTANT: Rust Detection - 660 nm (1st derivative)**

**This is the key rust signature!** The first derivative at 660 nm shows the rate of spectral change, which is highly sensitive to iron oxide (rust) on corroded metal surfaces.

In [ ]:
# 660 nm - RUST SIGNATURE (1st derivative)
intensity_660_1st = plot_single_wavelength_with_color(
    cube.data_corrected,
    cube.wavelengths,
    target_wavelength=660,
    derivative_order=1,
    window=2,
    title="660 nm - RUST SIGNATURE (1st Derivative)\n⚠️ Primary rust detection - rate of spectral change",
)

In [ ]:
# 660 nm - RUST SIGNATURE (1st derivative)
intensity_660_1st = plot_single_wavelength_with_color(
    cube.data_corrected,
    cube.wavelengths,
    target_wavelength=660,
    derivative_order=1,
    window=2,
    title="660 nm - RUST SIGNATURE (1st Derivative)\n⚠️ Primary rust detection - rate of spectral change",
)

In [ ]:
# 660 nm - RUST SIGNATURE (1st derivative)
intensity_660_1st = plot_single_wavelength_with_color(
    cube.data_corrected,
    cube.wavelengths,
    target_wavelength=660,
    derivative_order=0,
    window=2,
    title="660 nm - RUST SIGNATURE (1st Derivative)\n⚠️ Primary rust detection - rate of spectral change",
)

In [ ]:
# 660 nm - RUST SIGNATURE (1st derivative)
intensity_660_1st = plot_single_wavelength_with_color(
    cube.data_corrected,
    cube.wavelengths,
    target_wavelength=660,
    derivative_order=0,
    window=2,
    title="660 nm - RUST SIGNATURE (1st Derivative)\n⚠️ Primary rust detection - rate of spectral change",
)

In [ ]:
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=1258,
    crop_center_slit=212,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    # use_wavelength_colormap=True,
    # wavelength_colormap_target=660,
    # vmin=0.086,
    # vmax=0.1,
)
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=5592,
    crop_center_slit=765,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    # use_wavelength_colormap=True,
    # wavelength_colormap_target=660,
    # vmin=0.086,
    # vmax=0.1,
)
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=5160,
    crop_center_slit=613,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    #     # blue_wl=550
    #     use_wavelength_colormap=True,
    #     wavelength_colormap_target=660,
    #     vmin=0.086,
    #     vmax=0.1,
)
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=717,
    crop_center_slit=385,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    # use_wavelength_colormap=True,
    # wavelength_colormap_target=660,
    # vmin=0.086,
    # vmax=0.1,
)
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=4026,
    crop_center_slit=582,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    # use_wavelength_colormap=True,
    # wavelength_colormap_target=660,
    # vmin=0.086,
    # vmax=0.1,
)

In [ ]:
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=1258,
    crop_center_slit=212,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    use_wavelength_colormap=True,
    wavelength_colormap_target=660,
    vmin=0.086,
    vmax=0.1,
)
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=5592,
    crop_center_slit=765,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    use_wavelength_colormap=True,
    wavelength_colormap_target=660,
    vmin=0.086,
    vmax=0.1,
)
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=5160,
    crop_center_slit=613,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    use_wavelength_colormap=True,
    wavelength_colormap_target=660,
    vmin=0.086,
    vmax=0.1,
)
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=717,
    crop_center_slit=385,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    use_wavelength_colormap=True,
    wavelength_colormap_target=660,
    vmin=0.086,
    vmax=0.1,
)
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=4026,
    crop_center_slit=582,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    use_wavelength_colormap=True,
    wavelength_colormap_target=660,
    vmin=0.086,
    vmax=0.1,
)

In [ ]:
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=1258,
    crop_center_slit=212,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    use_wavelength_colormap=True,
    wavelength_colormap_target=660,
    # vmin=-0.000206,
    # vmax=0.000492,
    derivative_order=1,
    derivative_window=2,
)
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=5592,
    crop_center_slit=765,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    use_wavelength_colormap=True,
    wavelength_colormap_target=660,
    # vmin=0.086,
    derivative_order=1,
    # vmax=0.1,
    # vmin=-0.000206,
    # vmax=0.000492,
)
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=5160,
    crop_center_slit=613,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    use_wavelength_colormap=True,
    wavelength_colormap_target=660,
    derivative_order=1,
    # vmin=0.086,
    # vmin=-0.000206,
    # vmax=0.000492,
    # vmax=0.1,
)
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=717,
    crop_center_slit=385,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    use_wavelength_colormap=True,
    wavelength_colormap_target=660,
    derivative_order=1,
    # vmin=0.086,
    # vmin=-0.000206,
    # vmax=0.000492,
    # vmax=0.1,
)

In [ ]:
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=1258,
    crop_center_slit=212,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    use_wavelength_colormap=True,
    wavelength_colormap_target=660,
    vmin=-0.000206,
    vmax=0.000427,
    derivative_order=1,
    derivative_window=2,
)
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=5592,
    crop_center_slit=765,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    use_wavelength_colormap=True,
    wavelength_colormap_target=660,
    # vmin=0.086,
    derivative_order=1,
    # vmax=0.1,
    vmin=-0.000206,
    # vmax=0.000492,
    vmax=0.000427,
)
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=5160,
    crop_center_slit=613,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    use_wavelength_colormap=True,
    wavelength_colormap_target=660,
    derivative_order=1,
    # vmin=0.086,
    vmin=-0.000206,
    # vmax=0.000492,
    vmax=0.000427,
    # vmax=0.1,
)
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=717,
    crop_center_slit=385,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    use_wavelength_colormap=True,
    wavelength_colormap_target=660,
    derivative_order=1,
    # vmin=0.086,
    vmin=-0.000206,
    # vmax=0.000492,
    vmax=0.000427,
    # vmax=0.1,
)

In [ ]:
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=1258,
    crop_center_slit=212,
    crop_width=400,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    use_wavelength_colormap=True,
    wavelength_colormap_target=595,
)

### Wavelength Series: 490-700 nm (10 nm intervals)

Generate plots for all wavelengths from 490 to 700 nm with 10 nm intervals.
This creates 22 plots showing the spectral variation across the visible-to-NIR range.

In [ ]:
# Generate plots for wavelengths from 490 to 700 nm with 10 nm intervals
import time

wavelengths_to_plot = range(490, 710, 10)  # 490, 500, 510, ..., 700
total_plots = len(list(wavelengths_to_plot))

print(f"🎨 Generating {total_plots} wavelength plots...")
print(f"   Wavelengths: {list(wavelengths_to_plot)} nm")
print(f"   This will take a few moments...\n")

start_time = time.time()

for i, wl in enumerate(wavelengths_to_plot, 1):
    print(f"[{i}/{total_plots}] Plotting {wl} nm...")

    cube.plot_rgb(
        use_corrected=True,
        flip_axes=True,
        flip_horizontal=True,
        crop_center_track=1258,
        crop_center_slit=212,
        crop_width=500,
        show_file_boundaries=False,
        crop_aspect_ratio=3.5,
        use_wavelength_colormap=True,
        wavelength_colormap_target=wl,
    )

elapsed = time.time() - start_time
print(f"\n✅ All {total_plots} plots generated in {elapsed:.1f} seconds!")
print(f"   Average: {elapsed/total_plots:.2f} seconds per plot")

In [ ]:
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=1258,
    crop_center_slit=212,
    crop_width=400,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    # red_wl=678,
    # green_wl=595,
    # blue_wl=550
    use_wavelength_colormap=True,
    wavelength_colormap_target=660,
    derivative_order=1,
)

In [ ]:
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=1258,
    crop_center_slit=212,
    crop_width=400,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    red_wl=678,
    green_wl=595,
    blue_wl=550,
)

In [ ]:
# 660 nm - RUST SIGNATURE (1st derivative)
intensity_660_1st = plot_single_wavelength_with_color(
    cube.data_corrected,
    cube.wavelengths,
    target_wavelength=660,
    derivative_order=1,
    window=2,
    title="660 nm - RUST SIGNATURE (1st Derivative)\n⚠️ Primary rust detection - rate of spectral change",
)

In [ ]:
# 660 nm - RUST SIGNATURE (1st derivative)
intensity_660_1st = plot_single_wavelength_with_color(
    cube.data_corrected,
    cube.wavelengths,
    target_wavelength=660,
    derivative_order=1,
    window=3,
    title="660 nm - RUST SIGNATURE (1st Derivative)\n⚠️ Primary rust detection - rate of spectral change",
)

In [ ]:
# 660 nm - RUST SIGNATURE (1st derivative)
intensity_660_1st = plot_single_wavelength_with_color(
    cube.data_corrected,
    cube.wavelengths,
    target_wavelength=660,
    derivative_order=1,
    window=5,
    title="660 nm - RUST SIGNATURE (1st Derivative)\n⚠️ Primary rust detection - rate of spectral change",
)

### 4.8 Chlorophyll Absorption - 675 nm (0th derivative)

In [ ]:
# 675 nm - Chlorophyll absorption (0th derivative)
intensity_675_raw = plot_single_wavelength_with_color(
    cube.data_corrected,
    cube.wavelengths,
    target_wavelength=675,
    derivative_order=0,
    title="675 nm - Chlorophyll Absorption (Raw Intensity)\nBiofilm detection",
)

### 4.9 Chlorophyll Absorption - 675 nm (2nd derivative)

In [ ]:
# 675 nm - Chlorophyll absorption (2nd derivative for trough curvature)
intensity_675_2nd = plot_single_wavelength_with_color(
    cube.data_corrected,
    cube.wavelengths,
    target_wavelength=675,
    derivative_order=2,
    window=2,
    title="675 nm - Chlorophyll Absorption (2nd Derivative)\nRefined biofilm detection via curvature",
)

## 5. Compare Multiple Wavelengths in Grid

Create a grid comparison of all wavelengths for easier visual analysis.

In [ ]:
# Create grid comparison using wavelength-specific colors
target_wavelengths = [500, 550, 580, 600, 620, 660]
wavelength_labels = [
    "500 nm - Reference Green",
    "550 nm - Green Peak",
    "580 nm - Sediment/Iron",
    "600 nm - Rust Band",
    "620 nm - Cyanobacteria/UXO",
    "660 nm - RUST (1st deriv)",
]

# Plot grid with 0th derivative for first 5, 1st derivative for 660nm
intensities = plot_wavelength_grid_with_colors(
    cube.data_corrected,
    cube.wavelengths,
    target_wavelengths,
    wavelength_labels=wavelength_labels,
    derivative_order=0,  # Note: this will be updated for mixed derivatives
    title="Transect 028 - Single Wavelength Intensity Comparison\n(Natural wavelength colors: white = min, color = max)",
)

## 6. Derivative Analysis Comparison

Compare 0th, 1st, and 2nd derivatives for key wavelengths.

### 6.1 Test: 600 nm Rust Analysis (All Derivatives)

In [ ]:
# Create grid for 600 nm with all derivative orders using wavelength colors
fig, axes = plt.subplots(1, 3, figsize=(30, 8))

derivative_configs = [
    (0, "Raw Intensity"),
    (1, "1st Derivative (dI/dλ)"),
    (2, "2nd Derivative (d²I/dλ²)"),
]

for idx, (order, label) in enumerate(derivative_configs):
    # Compute derivative
    wl_idx = np.argmin(np.abs(cube.wavelengths - 600))
    window = 2

    if order == 0:
        intensity = cube.data_corrected[:, :, wl_idx]
    elif order == 1:
        intensity_forward = cube.data_corrected[:, :, wl_idx + window]
        intensity_backward = cube.data_corrected[:, :, wl_idx - window]
        wl_forward = cube.wavelengths[wl_idx + window]
        wl_backward = cube.wavelengths[wl_idx - window]
        intensity = (intensity_forward - intensity_backward) / (
            wl_forward - wl_backward
        )
    else:  # order == 2
        intensity_center = cube.data_corrected[:, :, wl_idx]
        intensity_forward = cube.data_corrected[:, :, wl_idx + window]
        intensity_backward = cube.data_corrected[:, :, wl_idx - window]
        h = cube.wavelengths[wl_idx + window] - cube.wavelengths[wl_idx]
        intensity = (intensity_forward - 2 * intensity_center + intensity_backward) / (
            h**2
        )

    # Normalize to [0, 1]
    intensity_min = np.nanmin(intensity)
    intensity_max = np.nanmax(intensity)
    if intensity_max > intensity_min:
        intensity_normalized = (intensity - intensity_min) / (
            intensity_max - intensity_min
        )
    else:
        intensity_normalized = np.zeros_like(intensity)

    # Create wavelength-specific colormap
    cmap = create_wavelength_colormap(600)

    im = axes[idx].imshow(
        intensity_normalized.T,
        aspect="auto",
        cmap=cmap,
        origin="lower",
        vmin=0,
        vmax=1,
    )

    axes[idx].set_title(f"600 nm Rust Band\n{label}", fontsize=14, fontweight="bold")
    axes[idx].set_xlabel("Track Index", fontsize=12)
    axes[idx].set_ylabel("Slit Index", fontsize=12)

    cbar = plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)
    cbar.set_label(label, fontsize=11)

plt.suptitle(
    "600 nm Rust Analysis - Derivative Comparison\n(Orange color intensity = spectral intensity)",
    fontsize=16,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

### 6.2 Test: 675 nm Chlorophyll Analysis (All Derivatives)

In [ ]:
# Create grid for 675 nm with all derivative orders using wavelength colors
fig, axes = plt.subplots(1, 3, figsize=(30, 8))

for idx, (order, label) in enumerate(derivative_configs):
    # Compute derivative
    wl_idx = np.argmin(np.abs(cube.wavelengths - 675))
    window = 2

    if order == 0:
        intensity = cube.data_corrected[:, :, wl_idx]
    elif order == 1:
        intensity_forward = cube.data_corrected[:, :, wl_idx + window]
        intensity_backward = cube.data_corrected[:, :, wl_idx - window]
        wl_forward = cube.wavelengths[wl_idx + window]
        wl_backward = cube.wavelengths[wl_idx - window]
        intensity = (intensity_forward - intensity_backward) / (
            wl_forward - wl_backward
        )
    else:  # order == 2
        intensity_center = cube.data_corrected[:, :, wl_idx]
        intensity_forward = cube.data_corrected[:, :, wl_idx + window]
        intensity_backward = cube.data_corrected[:, :, wl_idx - window]
        h = cube.wavelengths[wl_idx + window] - cube.wavelengths[wl_idx]
        intensity = (intensity_forward - 2 * intensity_center + intensity_backward) / (
            h**2
        )

    # Normalize to [0, 1]
    intensity_min = np.nanmin(intensity)
    intensity_max = np.nanmax(intensity)
    if intensity_max > intensity_min:
        intensity_normalized = (intensity - intensity_min) / (
            intensity_max - intensity_min
        )
    else:
        intensity_normalized = np.zeros_like(intensity)

    # Create wavelength-specific colormap
    cmap = create_wavelength_colormap(675)

    im = axes[idx].imshow(
        intensity_normalized.T,
        aspect="auto",
        cmap=cmap,
        origin="lower",
        vmin=0,
        vmax=1,
    )

    axes[idx].set_title(f"675 nm Chlorophyll\n{label}", fontsize=14, fontweight="bold")
    axes[idx].set_xlabel("Track Index", fontsize=12)
    axes[idx].set_ylabel("Slit Index", fontsize=12)

    cbar = plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)
    cbar.set_label(label, fontsize=11)

plt.suptitle(
    "675 nm Chlorophyll Analysis - Derivative Comparison\n(Red color intensity = spectral intensity)",
    fontsize=16,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

## 7. Summary

This notebook has demonstrated:
1. ✅ Loading transect 028 data
2. ✅ Applying complete preprocessing pipeline (illumination, smoothing, wavelength crop, L2 normalization)
3. ✅ **NEW: Wavelength-specific color visualization** (white = min, wavelength color = max)
4. ✅ Visualizing key wavelengths with natural colors:
   - 500 nm (cyan/green) - Reference baseline
   - 550 nm (green) - Biofilm/algae detection
   - 580 nm (yellow/orange) - Sediment/iron transition
   - 600 nm (orange) - Rust band
   - 620 nm (orange/red) - Cyanobacteria/UXO leakage
   - **660 nm (red) - PRIMARY RUST SIGNATURE (1st derivative)**
   - 675 nm (deep red) - Chlorophyll absorption
5. ✅ All visualization functions moved to `utils/ndi_analysis_utils.py`

**Key Innovation:**
- **Wavelength-specific colormaps**: Each wavelength is visualized with its natural color, creating intuitive heatmaps where color intensity directly represents spectral intensity

**Key Findings:**
- **660 nm 1st derivative**: Most sensitive rust detection method
- **0th derivative**: Shows raw intensity patterns
- **1st derivative**: Highlights rate of change (slopes) - best for rust
- **2nd derivative**: Emphasizes curvature (absorption/emission features)

**Functions in `utils/ndi_analysis_utils.py`:**
- `wavelength_to_rgb()`: Convert wavelength to natural RGB color
- `create_wavelength_colormap()`: Generate white-to-color colormap
- `plot_single_wavelength_with_color()`: Single wavelength visualization
- `plot_wavelength_grid_with_colors()`: Grid comparison with natural colors

**Next Steps:**
- Apply 660 nm 1st derivative for automated rust detection
- Combine with NDI analysis for multi-feature classification
- Use wavelength-specific visualizations for feature interpretation

## 8. NEW: Single-Wavelength Colormap Mode in plot_rgb()

**Major Feature:** `plot_rgb()` now supports single-wavelength visualization with natural colors!

This is a "master function" that can do both:
- **RGB composite mode** (default) - combines 3 wavelengths into RGB image
- **Single-wavelength colormap mode** (new!) - plots one wavelength with natural color

**Benefits:**
- All existing code using `plot_rgb()` continues to work
- All advanced features work in both modes: derivatives, cropping, flipping, ROIs, etc.
- Single-wavelength mode uses intuitive colormaps (white → wavelength color)

### 8.1 Test: Single Wavelength Mode (660 nm Rust Signature)

In [ ]:
# Test: plot_rgb() in single-wavelength mode with 660 nm (rust detection)
cube.plot_rgb(
    use_corrected=True,
    figsize=(20, 6),
    use_wavelength_colormap=True,
    wavelength_colormap_target=660,
    derivative_order=1,
    derivative_window=2,
)

### 8.2 Test: Single Wavelength with Cropping and Flipping

All advanced features work in single-wavelength mode!

In [ ]:
# Test: Single wavelength with all the advanced features
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    flip_horizontal=True,
    crop_center_track=1258,
    crop_center_slit=212,
    crop_width=400,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    use_wavelength_colormap=True,
    wavelength_colormap_target=660,
    derivative_order=1,
    derivative_window=2,
)

### 8.3 Comparison: plot_single_wavelength_with_color() vs plot_rgb() wavelength mode

Both functions should produce identical results!

In [ ]:
# Comparison: Old function vs new plot_rgb() wavelength mode
print("=" * 80)
print("OLD FUNCTION: plot_single_wavelength_with_color()")
print("=" * 80)
plot_single_wavelength_with_color(
    cube.data_corrected,
    cube.wavelengths,
    target_wavelength=600,
    derivative_order=0,
    title="600 nm - OLD FUNCTION",
    figsize=(20, 6),
)

print("\n" + "=" * 80)
print("NEW FUNCTION: cube.plot_rgb() with use_wavelength_colormap=True")
print("=" * 80)
cube.plot_rgb(
    use_corrected=True,
    figsize=(20, 6),
    use_wavelength_colormap=True,
    wavelength_colormap_target=600,
    derivative_order=0,
)

### 8.4 Summary: plot_rgb() Master Function

**Key Achievement:** `plot_rgb()` is now a true "master function" that can:

1. **RGB Composite Mode** (default):
   ```python
   cube.plot_rgb(use_corrected=True)  # Traditional RGB composite
   ```

2. **Single-Wavelength Colormap Mode** (new):
   ```python
   cube.plot_rgb(
       use_corrected=True,
       use_wavelength_colormap=True,
       wavelength_colormap_target=660
   )
   ```

**All Advanced Features Work in Both Modes:**
- ✅ Derivatives (0th, 1st, 2nd order)
- ✅ Cropping (crop_center_track, crop_center_slit, crop_width)
- ✅ Axis flipping (flip_axes, flip_horizontal, flip_vertical)
- ✅ ROIs (roi_collection)
- ✅ File boundaries
- ✅ Custom figure sizes

**Benefits:**
- All existing code continues to work (100% backward compatible)
- Single-wavelength mode produces same results as `plot_single_wavelength_with_color()`
- More powerful: has all the cropping/flipping features that the standalone function lacks
- Consistent API across both visualization modes

---


---


---


---


---


---


---


---


---


---


---


---


---


---


---


---


---


---


---


---


# wrapper

In [ ]:
# Define your crop regions
crop_regions = [
    {"track": 1258, "slit": 212, "label": "Region 1"},
    {"track": 5592, "slit": 765, "label": "Region 2"},
    {"track": 5160, "slit": 613, "label": "Region 3"},
    {"track": 717, "slit": 385, "label": "Region 4"},
    {"track": 4026, "slit": 582, "label": "Region 5"},
]

# Plot all crops with automatic global normalization
# 0th derivative (raw intensity)
results = plot_wavelength_crops(
    cube,
    crop_regions,
    wavelength_target=660,
    derivative_order=0,
    
    # derivative_window=2,
)

print(f"\n📋 Results:")
print(
    f"   Global normalization: vmin={results['vmin']:.6f}, vmax={results['vmax']:.6f}"
)
# # Same crops with 1st derivative (rate of spectral change - best for rust detection)
# results_1st = plot_wavelength_crops(
#     cube,
#     crop_regions,
#     wavelength_target=660,
#     derivative_order=1,
#     derivative_window=2,
# )

# print(f"\n📋 Results (1st derivative):")
# print(
#     f"   Global normalization: vmin={results_1st['vmin']:.6f}, vmax={results_1st['vmax']:.6f}"
# )